In [7]:
# 1. Clone YOLOv5 and install dependencies
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt
!pip install roboflow

import torch
from IPython.display import Image
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Cloning into 'yolov5'...
remote: Enumerating objects: 17777, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 17777 (delta 98), reused 55 (delta 54), pack-reused 17616 (from 2)
Receiving objects: 100% (17777/17777), 17.14 MiB | 24.72 MiB/s, done.
Resolving deltas: 100% (12076/12076), done.
/content/yolov5/yolov5
Setup complete. Using torch 2.9.0+cu126 (Tesla T4)


In [8]:
from roboflow import Roboflow
rf = Roboflow(api_key="X6WYoKFVxpfl6VptTbHq")
project = rf.workspace("infocom-project-a6jaj").project("apple-banana-orange-nhiie")
version = project.version(2)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to apple-banana-orange-2 in yolov5pytorch:: 100%|██████████| 1348/1348 [00:00<00:00, 3736.91it/s]


In [9]:
import os
import yaml

# 1. Chercher le dossier du dataset (qui commence par 'banana')
current_dir = os.getcwd()
folders = [d for d in os.listdir(current_dir) if os.path.isdir(d) and 'banana' in d]

if folders:
    dataset_dir = os.path.abspath(folders[0])
    original_yaml_path = os.path.join(dataset_dir, 'data.yaml')

    print(f"✅ Dossier trouvé : {dataset_dir}")

    # 2. Lire les noms des classes depuis le fichier original (pour ne pas se tromper d'ordre)
    if os.path.exists(original_yaml_path):
        with open(original_yaml_path, 'r') as f:
            old_data = yaml.safe_load(f)
            names = old_data.get('names', ['apple', 'banana', 'orange'])
            nc = old_data.get('nc', 3)
    else:
        # Fallback si le fichier n'existe pas
        names = ['apple', 'banana', 'orange']
        nc = 3

    # 3. Créer un NOUVEAU fichier de configuration "propre"
    # Roboflow nomme souvent le dossier validation "valid", YOLO cherche "val"
    new_yaml_content = {
        'path': dataset_dir,       # Chemin racine ABSOLU
        'train': 'train/images',   # Chemin relatif vers train
        'val': 'valid/images',     # Chemin relatif vers valid
        'nc': nc,
        'names': names
    }

    with open('fixed_data.yaml', 'w') as f:
        yaml.dump(new_yaml_content, f)

    print("\n📄 Fichier 'fixed_data.yaml' créé avec succès !")
    print(f"Classes détectées : {names}")

else:
    print("❌ Erreur : Impossible de trouver le dossier 'banana-apple-orange'. Vérifiez vos fichiers à gauche.")

✅ Dossier trouvé : /content/yolov5/yolov5/apple-banana-orange-2

📄 Fichier 'fixed_data.yaml' créé avec succès !
Classes détectées : ['apple', 'banana', 'orange']


In [10]:
# 3. Train the model
# We use the location from the downloaded dataset automatically
!python train.py --img 640 --batch 16 --epochs 50 --data {dataset.location}/data.yaml --weights yolov5s.pt --name fruit_model --cache

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2025-12-27 22:43:40.797356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766875420.818244    4562 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766875420.824412    4562 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766875420.840016    4562 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766875420.840048    4562 computation_placer.cc:177] computation placer already registere

In [1]:
import torch
import cv2
import numpy as np
import base64
import warnings
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image
from io import BytesIO

warnings.filterwarnings("ignore")

print("⏳ Chargement du modèle...")
model = torch.hub.load('ultralytics/yolov5', 'custom', path='runs/train/fruit_model/weights/best.pt', force_reload=True)

# 1. ON OUVRE LES VANNES (Seuil global très bas)
# On laisse tout passer au début, on filtrera nous-mêmes après.
model.conf = 0.10
model.iou = 0.45

# --- FONCTIONS VIDÉO (Inchangées) ---
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;
    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color: red; font-weight: bold;">👉 CLIQUEZ SUR LA VIDÉO POUR ARRÊTER</span>';
      div.appendChild(instruction);

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }
      var preCreate = Date.now();
      stream = await createDom();
      var preShow = Date.now();
      if (label != "") { labelElement.innerHTML = label; }
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;
      return {'create': preShow - preCreate, 'show': preCapture - preShow, 'capture': Date.now() - preCapture, 'img': result};
    }
    ''')
  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

# --- BOUCLE PRINCIPALE INTELLIGENTE ---
video_stream()
label_html = 'Démarrage...'
bbox = ''

while True:
    try:
        js_reply = video_frame(label_html, bbox)
        if not js_reply: break

        img = js_to_image(js_reply['img'])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 1. DÉTECTION BRUTE (Tout passe)
        results = model(img_rgb)

        # 2. FILTRAGE INTELLIGENT (C'est ici que la magie opère)
        # On récupère les boîtes (x1, y1, x2, y2, conf, class)
        detections = results.xyxy[0]
        good_detections = []

        if len(detections) > 0:
            for det in detections:
                conf = float(det[4]) # Confiance (ex: 0.85)
                cls_id = int(det[5]) # ID de la classe (0, 1, ou 2)
                label = results.names[cls_id] # Nom (apple, banana...)

                # --- VOS RÈGLES ---
                # Règle 1 : BANANE (Soyons gentils)
                # On accepte tout ce qui dépasse 10%
                if label == 'banana' and conf >= 0.10:
                    good_detections.append(det)

                # Règle 2 : POMME (Soyons sévères)
                # On refuse tout ce qui est en dessous de 80% (Adieu les visages !)
                elif label == 'apple' and conf >= 0.80:
                    good_detections.append(det)

                # Règle 3 : ORANGE (Standard)
                elif label == 'orange' and conf >= 0.30:
                    good_detections.append(det)

        # 3. On remplace les résultats par notre liste filtrée
        if len(good_detections) > 0:
            results.xyxy[0] = torch.stack(good_detections)
            status_text = "✅ Objets trouvés"
        else:
            # Si tout a été filtré (ex: visage rejeté), on vide la liste
            results.xyxy[0] = torch.tensor([])
            status_text = "⏳ Recherche..."

        # 4. Affichage
        annotated_img = results.render()[0]
        annotated_img_pil = PIL.Image.fromarray(annotated_img)
        io_buf = BytesIO()
        annotated_img_pil.save(io_buf, format='JPEG')
        bbox = 'data:image/jpeg;base64,{}'.format((b64encode(io_buf.getvalue())).decode('utf-8'))
        label_html = status_text

    except Exception as e:
        print(f"Erreur : {e}")
        break

ModuleNotFoundError: No module named 'google.colab'

In [15]:
# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create a folder in your Drive to keep things organized
import os
destination_folder = '/content/drive/MyDrive/My_YOLO_Project'
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# 3. Copy the trained model there
import shutil
# The path to your model based on our previous steps
source_file = 'runs/train/fruit_model/weights/best.pt'

if os.path.exists(source_file):
    shutil.copy(source_file, f'{destination_folder}/best.pt')
    print(f"✅ Success! Your model is saved at: {destination_folder}/best.pt")
else:
    print("❌ Error: Could not find the model file. Check the 'runs/train' folder.")

Mounted at /content/drive
✅ Success! Your model is saved at: /content/drive/MyDrive/My_YOLO_Project/best.pt


In [2]:
import torch
import cv2
import numpy as np
import base64
import warnings
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL.Image
from io import BytesIO
from google.colab import drive # <--- NOUVEAU

# 1. MONTER LE DRIVE
# Une fenêtre va s'ouvrir pour demander l'autorisation, cliquez sur "Connecter".
drive.mount('/content/drive')

warnings.filterwarnings("ignore")

print("⏳ Chargement du modèle depuis le Drive...")

# 2. CHARGER LE MODÈLE DU DRIVE
# Assurez-vous que le chemin ci-dessous correspond exactement à là où vous avez sauvegardé le fichier
# Si vous avez suivi mon conseil précédent, c'était dans "My_YOLO_Project"
model_path = '/content/drive/MyDrive/My_YOLO_Project/best.pt'

# Vérification de sécurité pour éviter le crash
import os
if not os.path.exists(model_path):
    print(f"❌ ERREUR : Le fichier n'est pas trouvé ici : {model_path}")
    print("👉 Vérifiez dans la barre de gauche (Dossier Drive) et copiez le bon chemin.")
else:
    model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=True)

    # 3. RÉGLAGES (Vos règles intelligentes)
    model.conf = 0.10
    model.iou = 0.45

    # --- FONCTIONS VIDÉO (Inchangées) ---
    def video_stream():
      js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;
        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
           stream.getVideoTracks()[0].stop();
           video.remove();
           div.remove();
           video = null;
           div = null;
           stream = null;
           imgElement = null;
           captureCanvas = null;
           labelElement = null;
        }

        function onAnimationFrame() {
          if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame);
          }
          if (pendingResolve) {
            var result = "";
            if (!shutdown) {
              captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
              result = captureCanvas.toDataURL('image/jpeg', 0.8)
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
          }
        }

        async function createDom() {
          if (div !== null) { return stream; }
          div = document.createElement('div');
          div.style.border = '2px solid black';
          div.style.padding = '3px';
          div.style.width = '100%';
          div.style.maxWidth = '600px';
          document.body.appendChild(div);

          const modelOut = document.createElement('div');
          modelOut.innerHTML = "<span>Status:</span>";
          labelElement = document.createElement('span');
          labelElement.innerText = 'No data';
          labelElement.style.fontWeight = 'bold';
          modelOut.appendChild(labelElement);
          div.appendChild(modelOut);

          video = document.createElement('video');
          video.style.display = 'block';
          video.width = div.clientWidth - 6;
          video.setAttribute('playsinline', '');
          video.onclick = () => { shutdown = true; };
          stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "environment"}});
          div.appendChild(video);

          imgElement = document.createElement('img');
          imgElement.style.position = 'absolute';
          imgElement.style.zIndex = 1;
          imgElement.onclick = () => { shutdown = true; };
          div.appendChild(imgElement);

          const instruction = document.createElement('div');
          instruction.innerHTML = '<span style="color: red; font-weight: bold;">👉 CLIQUEZ SUR LA VIDÉO POUR ARRÊTER</span>';
          div.appendChild(instruction);

          video.srcObject = stream;
          await video.play();

          captureCanvas = document.createElement('canvas');
          captureCanvas.width = 640;
          captureCanvas.height = 480;
          window.requestAnimationFrame(onAnimationFrame);

          return stream;
        }
        async function stream_frame(label, imgData) {
          if (shutdown) {
            removeDom();
            shutdown = false;
            return '';
          }
          var preCreate = Date.now();
          stream = await createDom();
          var preShow = Date.now();
          if (label != "") { labelElement.innerHTML = label; }
          if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
          }
          var preCapture = Date.now();
          var result = await new Promise(function(resolve, reject) {
            pendingResolve = resolve;
          });
          shutdown = false;
          return {'create': preShow - preCreate, 'show': preCapture - preShow, 'capture': Date.now() - preCapture, 'img': result};
        }
        ''')
      display(js)

    def video_frame(label, bbox):
      data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
      return data

    def js_to_image(js_reply):
      image_bytes = b64decode(js_reply.split(',')[1])
      jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
      img = cv2.imdecode(jpg_as_np, flags=1)
      return img

    # --- BOUCLE PRINCIPALE INTELLIGENTE ---
    video_stream()
    label_html = 'Démarrage...'
    bbox = ''

    while True:
        try:
            js_reply = video_frame(label_html, bbox)
            if not js_reply: break

            img = js_to_image(js_reply['img'])
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # 1. DÉTECTION BRUTE (Tout passe)
            results = model(img_rgb)

            # 2. FILTRAGE INTELLIGENT
            detections = results.xyxy[0]
            good_detections = []

            if len(detections) > 0:
                for det in detections:
                    conf = float(det[4])
                    cls_id = int(det[5])
                    label = results.names[cls_id]

                    # --- VOS RÈGLES ---
                    if label == 'banana' and conf >= 0.10:
                        good_detections.append(det)
                    elif label == 'apple' and conf >= 0.85:
                        good_detections.append(det)
                    elif label == 'orange' and conf >= 0.30:
                        good_detections.append(det)

            # 3. Remplacement des résultats
            if len(good_detections) > 0:
                results.xyxy[0] = torch.stack(good_detections)
                status_text = "✅ Objets trouvés"
            else:
                results.xyxy[0] = torch.tensor([])
                status_text = "⏳ Recherche..."

            # 4. Affichage
            annotated_img = results.render()[0]
            annotated_img_pil = PIL.Image.fromarray(annotated_img)
            io_buf = BytesIO()
            annotated_img_pil.save(io_buf, format='JPEG')
            bbox = 'data:image/jpeg;base64,{}'.format((b64encode(io_buf.getvalue())).decode('utf-8'))
            label_html = status_text

        except Exception as e:
            print(f"Erreur : {e}")
            break

ModuleNotFoundError: No module named 'google.colab'